# Field Scan : classifieur de maladies des feuilles de bananier (procédure complète)

**Entrée** : `feuilles_pret_perou.zip` (images préparées par `scripts/preparer_feuilles.py`, avec `index.csv`), dans `Mon Drive/field_scan/`.

**Ensembles** (carnet de données) :
- entraînement : Tanzanie 2021 (sans les 397 tiges) + Pérou 2026 (feuilles saines et Sigatoka) : 16 963 images ;
- validation : Tanzanie 2022-2023, un zip par classe : 3 001 images ;
- test : BananaLSD (Bangladesh, `OriginalSet`) : 897 images, dont Cordana et Pestalotiopsis, maladies **inconnues** ; **déjà consulté une fois** (modèle sans Pérou) : ses nouveaux résultats sont **indicatifs** ;
- test_perou : 505 Cordana péruviennes, jamais apprises ni utilisées pour un réglage : **premier test** du rejet de Cordana.

**Règles fixées avant tout résultat**
1. v1 : recadrages de 30 à 100 % ; meilleure époque = F1 macro de validation (images entières).
2. v2 : recadrages de 15 à 100 % et masquage aléatoire ; meilleure époque = moyenne des F1 macro (images entières et recadrages centrés).
3. Variantes (v1, v2, v1+v2, chacune avec ou sans 4 vues) : parmi celles qui n'augmentent pas les fausses alertes sur les feuilles saines par rapport à v2 seul, la meilleure moyenne des F1 (entières et recadrages).
4. Méthode de rejet : la meilleure, en moyenne sur trois inconnues simulées (une classe écartée à tour de rôle), du taux de rejet de l'inconnue quand 95 % des feuilles connues sont acceptées ; l'AUROC départage.
5. Seuil de rejet : 95 % des feuilles connues de la validation acceptées.
6. Les tests (section 9) ne sont lancés qu'une fois, tout étant figé.

Avant de commencer : **Exécution > Modifier le type d'exécution > T4 GPU**. Si la session se coupe, relancer les cellules dans l'ordre : avec `REPRENDRE = True`, les modèles déjà enregistrés sur Drive sont relus au lieu d'être réentraînés.

## 1. Données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
DOSSIER_DRIVE = '/content/drive/MyDrive/field_scan'
ZIP = f'{DOSSIER_DRIVE}/feuilles_pret_perou.zip'
DONNEES = '/content/feuilles'
SORTIE = f'{DOSSIER_DRIVE}/modeles_maladies_perou'
REPRENDRE = True                       # relire les modèles déjà enregistrés au lieu de les réentraîner
os.makedirs(SORTIE, exist_ok=True)
assert os.path.exists(ZIP), f"Introuvable : {ZIP}"

if not os.path.exists(f'{DONNEES}/index.csv'):
    with zipfile.ZipFile(ZIP) as z:          # « \ » remplacé par « / » si le zip vient de PowerShell
        for m in z.infolist():
            chemin = m.filename.replace('\\', '/')
            if chemin.endswith('/'):
                continue
            cible = os.path.join(DONNEES, chemin)
            os.makedirs(os.path.dirname(cible), exist_ok=True)
            with z.open(m) as src, open(cible, 'wb') as dst:
                shutil.copyfileobj(src, dst)

import pandas as pd
index = pd.read_csv(f'{DONNEES}/index.csv')
index = index[index.retenue].reset_index(drop=True)
assert all(os.path.exists(f'{DONNEES}/{c}') for c in index.chemin.sample(200, random_state=0)), "images manquantes"
print(index.groupby(['ensemble', 'classe']).size().unstack(fill_value=0))

## 2. Versions et reproductibilité

In [ ]:
import torch, torchvision, platform, random, time, json, numpy as np
GRAINE = 2026
def graine():
    random.seed(GRAINE); np.random.seed(GRAINE); torch.manual_seed(GRAINE)
graine()
VERSIONS = dict(torch=torch.__version__, torchvision=torchvision.__version__, python=platform.python_version(),
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun')
print(VERSIONS)
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > T4 GPU"
DEVICE = 'cuda'

## 3. Jeux de données et augmentations

- v1 : recadrage aléatoire de 30 à 100 % de l'image ; v2 : de 15 à 100 %, plus un masquage aléatoire (contre le raccourci de la nervure centrale, observé sur les cartes d'attention de v1).
- Communs : retournements, couleurs, flou léger, recompression JPEG aléatoire (appareils différents).
- Évaluation : image entière (petit côté à 224, centre) et recadrage centré (60 % du centre), pour détecter un modèle qui s'appuie sur le fond.

In [ ]:
import io
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

CLASSES = ['cercosporiose_noire', 'fusariose', 'saine']
MOYENNE, ECART = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
NORMALISATION = T.Normalize(MOYENNE, ECART)
moy, ec = torch.tensor(MOYENNE)[:, None, None], torch.tensor(ECART)[:, None, None]   # pour réafficher les images

class JPEGAleatoire:
    '''Recompression JPEG de qualité aléatoire, pour imiter des appareils différents.'''
    def __init__(self, p=0.5, qualites=(30, 95)): self.p, self.q = p, qualites
    def __call__(self, im):
        if random.random() > self.p: return im
        b = io.BytesIO(); im.save(b, 'JPEG', quality=random.randint(*self.q)); b.seek(0)
        return Image.open(b).convert('RGB')

class RecadrageCentre:
    def __init__(self, part): self.part = part
    def __call__(self, im):
        l, h = im.size; cl, ch = int(l * self.part), int(h * self.part)
        return im.crop(((l - cl) // 2, (h - ch) // 2, (l + cl) // 2, (h + ch) // 2))

def augmentations(recadrage_min, masquage):
    etapes = [JPEGAleatoire(), T.RandomResizedCrop(224, scale=(recadrage_min, 1.0)), T.RandomHorizontalFlip(),
              T.RandomVerticalFlip(), T.ColorJitter(0.3, 0.3, 0.3, 0.03),
              T.RandomApply([T.GaussianBlur(5, (0.1, 1.5))], p=0.2), T.ToTensor(), NORMALISATION]
    if masquage: etapes.append(T.RandomErasing(p=0.25, scale=(0.02, 0.15)))
    return T.Compose(etapes)

T_V1, T_V2 = augmentations(0.3, False), augmentations(0.15, True)
T_ENTIERE = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(), NORMALISATION])
T_CENTRE = T.Compose([RecadrageCentre(0.6), T.Resize(224), T.CenterCrop(224), T.ToTensor(), NORMALISATION])

class Feuilles(Dataset):
    '''Images de l'index ; étiquette = rang dans `classes`, ou -1 pour une classe inconnue.'''
    def __init__(self, lignes, transformation, classes=CLASSES):
        self.lignes, self.t, self.classes = lignes.reset_index(drop=True), transformation, classes
    def __len__(self): return len(self.lignes)
    def __getitem__(self, i):
        r = self.lignes.iloc[i]
        y = self.classes.index(r.classe) if r.classe in self.classes else -1
        return self.t(Image.open(f'{DONNEES}/{r.chemin}').convert('RGB')), y

def lignes(ensemble, classes=None):
    l = index[index.ensemble == ensemble]
    return l if classes is None else l[l.classe.isin(classes)]

def chargeur(lignes_, transformation, classes=CLASSES, melanger=False, taille=64):
    return DataLoader(Feuilles(lignes_, transformation, classes), batch_size=taille, shuffle=melanger,
                      num_workers=2, pin_memory=True, drop_last=melanger)

L_VAL = chargeur(lignes('validation'), T_ENTIERE)
L_VAL_CENTRE = chargeur(lignes('validation'), T_CENTRE)
print(len(lignes('entrainement')), 'images d’entraînement |', len(L_VAL.dataset), 'de validation')

In [ ]:
# Contrôle visuel : ce que voit le modèle pendant l'entraînement (recette v2)
import matplotlib.pyplot as plt
x, y = next(iter(chargeur(lignes('entrainement'), T_V2, melanger=True)))
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for ax, im, c in zip(axes.flat, x[:12], y[:12]):
    ax.imshow((im * ec + moy).clamp(0, 1).permute(1, 2, 0)); ax.set_title(CLASSES[c], fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Modèle, prédiction, mesures et entraînement

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.metrics import f1_score, confusion_matrix

def creer_modele(n_classes=len(CLASSES), pre_entraine=True):
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pre_entraine else None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, n_classes)
    return m

def caracteristiques_et_logits(m, x):
    f = torch.flatten(m.avgpool(m.features(x)), 1)
    return f, m.classifier(f)

@torch.no_grad()
def predire(m, chargeur_):
    m.eval(); F, L, Y = [], [], []
    for x, y in chargeur_:
        with torch.autocast('cuda', dtype=torch.float16):
            f, l = caracteristiques_et_logits(m, x.to(DEVICE, non_blocking=True))
        F.append(f.float().cpu()); L.append(l.float().cpu()); Y.append(y)
    return torch.cat(F), torch.cat(L), torch.cat(Y)

def f1(L, Y):
    return round(float(f1_score(Y.numpy(), L.argmax(1).numpy(), average='macro')), 4)

def bilan(L, Y):
    m_ = confusion_matrix(Y.numpy(), L.argmax(1).numpy(), labels=range(len(CLASSES)))
    d = {f'rappel_{c}': round(float(m_[k, k] / m_[k].sum()), 3) for k, c in enumerate(CLASSES)}
    s = CLASSES.index('saine')
    d['fausses_alertes_saines'] = round(float((m_[s].sum() - m_[s, s]) / m_[s].sum()), 3)
    d['exactitude'] = round(float((L.argmax(1) == Y).float().mean()), 4); d['f1_macro'] = f1(L, Y)
    return d

PERTE = nn.CrossEntropyLoss(label_smoothing=0.1)
RECETTE = dict(epoques=12, lot=64, lr=3e-4, decroissance_poids=1e-4, lissage_etiquettes=0.1, programme='cosinus')

def entrainer(nom, transformation, critere, classes=CLASSES, lignes_entr=None, epoques=RECETTE['epoques']):
    '''critere : 'entieres' (F1 validation, images entières), 'moyenne' (moyenne entières / recadrages centrés),
    ou None (pas de choix d'époque : on garde la dernière ; utilisé pour les inconnues simulées).'''
    chemin, chemin_h = f'{SORTIE}/meilleur_{nom}.pt', f'{SORTIE}/historique_{nom}.json'
    m = creer_modele(len(classes)).to(DEVICE)
    if REPRENDRE and os.path.exists(chemin) and os.path.exists(chemin_h):
        m.load_state_dict(torch.load(chemin)); h = json.load(open(chemin_h))
        print(f'{nom} : repris depuis Drive (époque retenue {h["epoque_retenue"]})'); return m, h
    graine()
    L_tr = chargeur(lignes('entrainement', classes) if lignes_entr is None else lignes_entr, transformation, classes, melanger=True)
    opt = torch.optim.AdamW(m.parameters(), lr=RECETTE['lr'], weight_decay=RECETTE['decroissance_poids'])
    prog = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epoques * len(L_tr))
    ech = torch.amp.GradScaler('cuda'); historique, meilleur = [], (-1, None)
    for ep in range(1, epoques + 1):
        m.train(); debut, total, n = time.time(), 0.0, 0
        for x, y in L_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                perte = PERTE(m(x), y)
            ech.scale(perte).backward(); ech.step(opt); ech.update(); prog.step()
            total += perte.item() * len(y); n += len(y)
        ligne = dict(epoque=ep, perte=round(total / n, 4))
        if critere is not None:
            _, Le, Ye = predire(m, L_VAL); ligne['f1_entieres'] = f1(Le, Ye)
            if critere == 'moyenne':
                _, Lr, Yr = predire(m, L_VAL_CENTRE); ligne['f1_recadrages'] = f1(Lr, Yr)
            ligne['critere'] = ligne['f1_entieres'] if critere == 'entieres' else round((ligne['f1_entieres'] + ligne['f1_recadrages']) / 2, 4)
        ligne['duree_s'] = round(time.time() - debut); historique.append(ligne); print(nom, ligne)
        valeur = ligne.get('critere', ep)          # sans critère : la dernière époque l'emporte
        if valeur > meilleur[0]:
            meilleur = (valeur, ep); torch.save(m.state_dict(), chemin)
    h = dict(historique=historique, epoque_retenue=meilleur[1], critere=critere)
    json.dump(h, open(chemin_h, 'w'), indent=2)
    m.load_state_dict(torch.load(chemin))
    print(f'{nom} : époque retenue {meilleur[1]}'); return m, h

## 5. Entraînement de v1 et v2 (environ 35 minutes chacun)

In [ ]:
modele_v1, H_V1 = entrainer('v1', T_V1, 'entieres')
modele_v2, H_V2 = entrainer('v2', T_V2, 'moyenne')

## 6. Choix de la variante (validation seulement)

Quatre vues d'une même image (une feuille vue d'en haut n'a pas de sens fixe), et combinaison des deux modèles par moyenne des sorties.

In [ ]:
VUES = {'identite': lambda x: x, 'miroir_horizontal': lambda x: torch.flip(x, [3]),
        'miroir_vertical': lambda x: torch.flip(x, [2]), 'rotation_180': lambda x: torch.flip(x, [2, 3])}
MODELES = {'v1': modele_v1, 'v2': modele_v2}

@torch.no_grad()
def sorties_par_vue(m, chargeur_):
    '''Logits pour chaque vue, caractéristiques de la vue identité, étiquettes.'''
    m.eval(); L = {v: [] for v in VUES}; F, Y = [], []
    for x, y in chargeur_:
        x = x.to(DEVICE, non_blocking=True)
        for v, f in VUES.items():
            with torch.autocast('cuda', dtype=torch.float16):
                fe, lo = caracteristiques_et_logits(m, f(x))
            L[v].append(lo.float().cpu())
            if v == 'identite': F.append(fe.float().cpu())
        Y.append(y)
    return {v: torch.cat(l) for v, l in L.items()}, torch.cat(F), torch.cat(Y)

def combiner(S, noms, plusieurs_vues):
    '''S[nom] = sorties_par_vue(...) ; moyenne des logits, caractéristiques mises bout à bout.'''
    vues = list(VUES) if plusieurs_vues else ['identite']
    L = torch.stack([S[n][0][v] for n in noms for v in vues]).mean(0)
    return torch.cat([S[n][1] for n in noms], 1), L, S[noms[0]][2]

S_VAL = {n: sorties_par_vue(m, L_VAL) for n, m in MODELES.items()}
S_VAL_CENTRE = {n: sorties_par_vue(m, L_VAL_CENTRE) for n, m in MODELES.items()}
lignes_var = []
for noms in (['v1'], ['v2'], ['v1', 'v2']):
    for vues in (False, True):
        _, Le, Ye = combiner(S_VAL, noms, vues); _, Lr, Yr = combiner(S_VAL_CENTRE, noms, vues)
        e, r = bilan(Le, Ye), bilan(Lr, Yr)
        lignes_var.append(dict(variante='+'.join(noms) + (' + 4 vues' if vues else ''), noms=noms, plusieurs_vues=vues,
                               f1_entieres=e['f1_macro'], f1_recadrages=r['f1_macro'],
                               critere=round((e['f1_macro'] + r['f1_macro']) / 2, 4), rappel_fusariose=e['rappel_fusariose'],
                               fa_saines_entieres=e['fausses_alertes_saines'], fa_saines_recadrages=r['fausses_alertes_saines']))
VARIANTES = pd.DataFrame(lignes_var)
print(VARIANTES.drop(columns=['noms', 'plusieurs_vues']).to_string(index=False))
REF = VARIANTES[VARIANTES.variante == 'v2'].iloc[0]
eligibles = VARIANTES[(VARIANTES.fa_saines_entieres <= REF.fa_saines_entieres) & (VARIANTES.fa_saines_recadrages <= REF.fa_saines_recadrages)]
CHOIX = eligibles.sort_values('critere', ascending=False).iloc[0]
NOMS_RETENUS, VUES_RETENUES = list(CHOIX.noms), bool(CHOIX.plusieurs_vues)
VARIANTES.drop(columns=['noms']).to_csv(f'{SORTIE}/variantes_validation.csv', index=False)
print(f'\nVariante retenue : {CHOIX.variante}')
_, Lv, Yv = combiner(S_VAL, NOMS_RETENUS, VUES_RETENUES); _, Lc, Yc = combiner(S_VAL_CENTRE, NOMS_RETENUS, VUES_RETENUES)
VALIDATION = dict(images_entieres=bilan(Lv, Yv), recadrages_centres=bilan(Lc, Yc))
print(VALIDATION)
print(pd.DataFrame(confusion_matrix(Yv.numpy(), Lv.argmax(1).numpy(), labels=range(len(CLASSES))),
                   index=[f'vrai {c}' for c in CLASSES], columns=[f'prédit {c}' for c in CLASSES]))

In [ ]:
# Cartes d'attention (Grad-CAM) de v1 et v2 sur les mêmes feuilles de validation
import matplotlib.pyplot as plt
def gradcam(m, x, classe):
    '''Zones de l'image qui ont le plus pesé dans la décision pour `classe`.'''
    m.eval(); stock = {}
    def capturer(module, entree, sortie):           # ne renvoie rien : ne modifie pas le réseau
        stock['a'] = sortie
        sortie.register_hook(lambda g: stock.__setitem__('g', g))
    crochet = m.features[-1].register_forward_hook(capturer)
    try:
        sortie = m(x)
    finally:
        crochet.remove()                             # retiré même si quelque chose échoue
    m.zero_grad(); sortie[0, classe].backward()
    poids = stock['g'].mean(dim=(2, 3), keepdim=True)
    carte = torch.relu((poids * stock['a']).sum(1))[0]
    return (carte / (carte.max() + 1e-8)).detach().cpu().numpy()

val = lignes('validation')
exemples = pd.concat([val[val.classe == c].sample(3, random_state=GRAINE) for c in CLASSES])
fig, axes = plt.subplots(3, 9, figsize=(22, 8))
for k, (_, r) in enumerate(exemples.iterrows()):
    x = T_ENTIERE(Image.open(f'{DONNEES}/{r.chemin}').convert('RGB'))[None].to(DEVICE)
    vue = (x[0].cpu() * ec + moy).clamp(0, 1).permute(1, 2, 0)
    axes[0, k].imshow(vue); axes[0, k].set_title(f'vrai : {r.classe[:12]}', fontsize=8); axes[0, k].axis('off')
    for ligne, nom in enumerate(MODELES, start=1):
        with torch.no_grad(): p_ = MODELES[nom](x).argmax(1).item()
        carte = gradcam(MODELES[nom], x.clone().requires_grad_(True), p_)
        axes[ligne, k].imshow(vue)
        axes[ligne, k].imshow(np.array(Image.fromarray((carte * 255).astype('uint8')).resize((224, 224))), alpha=0.5, cmap='jet')
        axes[ligne, k].set_title(f'{nom} : {CLASSES[p_][:12]}', fontsize=8); axes[ligne, k].axis('off')
plt.tight_layout(); plt.savefig(f'{SORTIE}/gradcam_v1_v2.png', dpi=110); plt.show()

## 7. Choix de la méthode de rejet : inconnues simulées (sans BananaLSD)

Pour chaque classe : un modèle entraîné sur les deux autres (recette v2, 6 époques, dernière époque gardée) ; la classe écartée, prise dans la **validation**, joue la maladie inconnue. Quatre méthodes de « familiarité » (plus haut = plus proche des classes connues) : énergie, probabilité maximale, Mahalanobis, plus proches voisins (similarité cosinus avec la 10e image d'entraînement la plus proche ; banque de 6 000 images au plus, tirées avec une graine fixe, comme en production).

In [ ]:
from sklearn.metrics import roc_auc_score
import torch.nn.functional as Fn
K_VOISINS, TAILLE_BANQUE, ACCEPTATION_CIBLE = 10, 6000, 0.95

def ajuster_rejet(F_banque, Y_banque, n):
    '''Moyennes et covariance (Mahalanobis), banque normalisée (voisins), apprises sur les feuilles d'entraînement.'''
    moy_ = torch.stack([F_banque[Y_banque == c].mean(0) for c in range(n)])
    cen = torch.cat([F_banque[Y_banque == c] - moy_[c] for c in range(n)])
    prec = torch.linalg.inv(cen.T @ cen / len(cen) + 1e-3 * torch.eye(F_banque.shape[1]))
    g = torch.Generator().manual_seed(GRAINE)
    choix = torch.randperm(len(F_banque), generator=g)[:TAILLE_BANQUE]
    return dict(moyennes=moy_, precision=prec, banque=Fn.normalize(F_banque[choix], dim=1), n=n)

def familiarite(p, F, L):
    d = torch.stack([((F - p['moyennes'][c]) @ p['precision'] * (F - p['moyennes'][c])).sum(1) for c in range(p['n'])], 1)
    Q, B = Fn.normalize(F, dim=1).to(DEVICE), p['banque'].to(DEVICE)
    voisins = torch.cat([torch.topk(Q[i:i + 1024] @ B.T, K_VOISINS, dim=1).values[:, -1].cpu() for i in range(0, len(Q), 1024)])
    return dict(energie=torch.logsumexp(L, 1), proba_max=torch.softmax(L, 1).max(1).values,
                mahalanobis=-d.min(1).values, plus_proches_voisins=voisins)

lignes_simu = []
for inconnue in CLASSES:
    connues = [c for c in CLASSES if c != inconnue]
    m, _ = entrainer(f'simu_sans_{inconnue}', T_V2, None, classes=connues, epoques=6)
    Fb, Lb, Yb = predire(m, chargeur(lignes('entrainement', connues), T_ENTIERE, connues))
    Fc, Lc_, Yc_ = predire(m, chargeur(lignes('validation', connues), T_ENTIERE, connues))
    Fi, Li, _ = predire(m, chargeur(lignes('validation', [inconnue]), T_ENTIERE, connues))
    p = ajuster_rejet(Fb, Yb, len(connues))
    sc, si = familiarite(p, Fc, Lc_), familiarite(p, Fi, Li)
    for meth in sc:
        seuil = torch.quantile(sc[meth], 1 - ACCEPTATION_CIBLE)
        etiq = np.r_[np.ones(len(sc[meth])), np.zeros(len(si[meth]))]
        lignes_simu.append(dict(inconnue=inconnue, methode=meth, rejet_a_95=round(float((si[meth] < seuil).float().mean()), 3),
                                auroc=round(float(roc_auc_score(etiq, np.r_[sc[meth].numpy(), si[meth].numpy()])), 3),
                                exactitude_connues=round(float((Lc_.argmax(1) == Yc_).float().mean()), 3)))
    del m; torch.cuda.empty_cache()
SIMU = pd.DataFrame(lignes_simu); SIMU.to_csv(f'{SORTIE}/inconnues_simulees.csv', index=False)
for col in ('rejet_a_95', 'auroc'):
    print(f'\n{col} :'); print(SIMU.pivot(index='methode', columns='inconnue', values=col).assign(moyenne=lambda d: d.mean(1).round(3)))
synthese = SIMU.groupby('methode')[['rejet_a_95', 'auroc']].mean().sort_values(['rejet_a_95', 'auroc'], ascending=False)
METHODE_REJET = synthese.index[0]
print(f'\nMéthode de rejet retenue (règle fixée à l’avance) : {METHODE_REJET}')

## 8. Rejet appliqué à la variante retenue, export ONNX et description

In [ ]:
!pip -q install onnx onnxruntime onnxscript
import onnxruntime as ort

# Seuils sur la validation (95 % des feuilles connues acceptées), pour la variante retenue
L_ENTR_EVAL = chargeur(lignes('entrainement'), T_ENTIERE)
P_ENTR = {n: predire(MODELES[n], L_ENTR_EVAL) for n in NOMS_RETENUS}      # vue identité suffit pour le rejet
Ft, Yt = torch.cat([P_ENTR[n][0] for n in NOMS_RETENUS], 1), P_ENTR[NOMS_RETENUS[0]][2]
PARAMS_REJET = ajuster_rejet(Ft, Yt, len(CLASSES))
Fv, Lv, Yv = combiner(S_VAL, NOMS_RETENUS, VUES_RETENUES)
fam_val = familiarite(PARAMS_REJET, Fv, Lv)
SEUILS = {k: float(torch.quantile(v, 1 - ACCEPTATION_CIBLE)) for k, v in fam_val.items()}
print('Seuils :', SEUILS)
for k, v in fam_val.items():
    acc = v >= SEUILS[k]
    print(f'{k:22s} acceptées {acc.float().mean():.3f} | exactitude sur les acceptées {(Lv.argmax(1)[acc] == Yv[acc]).float().mean():.3f}')

class AvecCaracteristiques(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        f = torch.flatten(self.m.avgpool(self.m.features(x)), 1)
        return self.m.classifier(f), f

options = dict(input_names=['image'], output_names=['logits', 'caracteristiques'], opset_version=17,
               dynamic_axes={'image': {0: 'lot'}, 'logits': {0: 'lot'}, 'caracteristiques': {0: 'lot'}})
x_reel = next(iter(L_VAL))[0][:8]
ONNX, sessions = {}, {}
for nom in NOMS_RETENUS:
    m_cpu = creer_modele(); m_cpu.load_state_dict(torch.load(f'{SORTIE}/meilleur_{nom}.pt', map_location='cpu'))
    enveloppe = AvecCaracteristiques(m_cpu).eval()      # l'enveloppe en mode évaluation (sinon l'export restaure le mode entraînement)
    chemin = f'{SORTIE}/maladies_{nom}.onnx'
    try:
        torch.onnx.export(enveloppe, torch.randn(1, 3, 224, 224), chemin, dynamo=False, **options)
    except TypeError:
        torch.onnx.export(enveloppe, torch.randn(1, 3, 224, 224), chemin, **options)
    enveloppe.eval()
    sessions[nom] = ort.InferenceSession(chemin, providers=['CPUExecutionProvider'])
    with torch.no_grad(): ref = enveloppe(x_reel)[0].numpy()
    sortie_onnx = sessions[nom].run(None, {'image': x_reel.numpy()})[0]
    ONNX[nom] = dict(fichier=os.path.basename(chemin), taille_mo=round(os.path.getsize(chemin) / 1e6, 1),
                     ecart_max=float(np.abs(sortie_onnx - ref).max()), memes_classes=bool((sortie_onnx.argmax(1) == ref.argmax(1)).all()))
print('ONNX :', ONNX)
x1 = torch.randn(1, 3, 224, 224); vues_ = list(VUES.values()) if VUES_RETENUES else [VUES['identite']]
debut = time.time()
for _ in range(10):
    for s in sessions.values():
        for f in vues_: s.run(None, {'image': f(x1).numpy()})
MS_PAR_IMAGE = round((time.time() - debut) / 10 * 1000)
print(f'Temps par image sur le processeur de Colab : {MS_PAR_IMAGE} ms')

np.savez(f'{SORTIE}/rejet.npz', moyennes=PARAMS_REJET['moyennes'].numpy(), precision=PARAMS_REJET['precision'].numpy(),
         banque=PARAMS_REJET['banque'].numpy().astype(np.float16))
json.dump(dict(versions=VERSIONS, classes=CLASSES, recette=RECETTE, historiques=dict(v1=H_V1, v2=H_V2),
               variante=CHOIX.variante, modeles=NOMS_RETENUS, vues=list(VUES) if VUES_RETENUES else ['identite'],
               combinaison='moyenne des logits sur les modèles et les vues ; caractéristiques : vue identité des modèles, mises bout à bout',
               variantes_validation=VARIANTES.drop(columns=['noms']).to_dict('records'), validation=VALIDATION,
               normalisation=dict(moyenne=MOYENNE, ecart_type=ECART),
               rejet=dict(methode=METHODE_REJET, acceptation_cible=ACCEPTATION_CIBLE, seuils=SEUILS, k_voisins=K_VOISINS,
                          taille_banque=int(len(PARAMS_REJET['banque'])), inconnues_simulees=SIMU.to_dict('records')),
               onnx=ONNX, ms_par_image_cpu_colab=MS_PAR_IMAGE),
          open(f'{SORTIE}/entrainement_maladies.json', 'w'), indent=2, ensure_ascii=False)
print(sorted(os.listdir(SORTIE)))

## 9. Tests : une seule fois, tout étant figé

- **test_perou** (505 Cordana du Pérou) : premier test, jamais consulté. Mesure le rejet de Cordana, et la part déclarée saine.
- **BananaLSD** : déjà consulté avec le modèle sans Pérou ; résultat **indicatif**, à comparer à l'ancien.
- Ablations : v1 seul et v2 seul, une vue, rejet par l'énergie.

Passer `EVALUER_TESTS = True` seulement quand tout est figé.

In [ ]:
EVALUER_TESTS = False

def mesurer(L, score, seuil, vrai):
    '''Mesures du sujet ; une image sous le seuil de familiarité reçoit « je ne sais pas ».'''
    pred = np.array([CLASSES[k] for k in L.argmax(1).tolist()])
    rep = np.where(score.numpy() >= seuil, pred, 'je_ne_sais_pas')
    inconnue = pd.Series(vrai).str.startswith('inconnue').to_numpy()
    r = dict(rejet_inconnues=(rep[inconnue] == 'je_ne_sais_pas').mean() if inconnue.any() else np.nan,
             inconnues_declarees_saines=(rep[inconnue] == 'saine').mean() if inconnue.any() else np.nan)
    for c in ('inconnue_cordana', 'inconnue_pestalotiopsis'):
        if (vrai == c).any(): r[f'rejet_{c[9:]}'] = (rep[vrai == c] == 'je_ne_sais_pas').mean()
    if (~inconnue).any():
        r['rejet_connues'] = (rep[~inconnue] == 'je_ne_sais_pas').mean()
        r['fausses_alertes_saines'] = np.isin(rep[vrai == 'saine'], ['cercosporiose_noire', 'fusariose']).mean()
        for c in ('saine', 'cercosporiose_noire'):
            vp = ((rep == c) & (vrai == c)).sum()
            r[f'precision_{c}'] = vp / max((rep == c).sum(), 1); r[f'rappel_{c}'] = vp / max((vrai == c).sum(), 1)
    r['predictions_fusariose'] = int((rep == 'fusariose').sum())
    return {k: (round(float(v), 3) if not isinstance(v, int) else v) for k, v in r.items()}, rep

if EVALUER_TESTS:
    RESULTATS, TABLEAUX = {}, {}
    for ens in ('test_perou', 'test'):
        lt = lignes(ens).reset_index(drop=True); vrai = lt.classe.to_numpy(); L_T = chargeur(lt, T_ENTIERE)
        S_T = {n: sorties_par_vue(MODELES[n], L_T) for n in MODELES}
        F_t, L_t, _ = combiner(S_T, NOMS_RETENUS, VUES_RETENUES)
        fam = familiarite(PARAMS_REJET, F_t, L_t)
        for meth in fam:
            RESULTATS[f'{ens} | {CHOIX.variante}, {meth}'], rep = mesurer(L_t, fam[meth], SEUILS[meth], vrai)
            if meth == METHODE_REJET:
                TABLEAUX[ens] = pd.crosstab(pd.Series(vrai, name='vrai'), pd.Series(rep, name='réponse'))
        for nom in MODELES:                           # ablations : un modèle seul, une vue, énergie
            seuil = float(torch.quantile(torch.logsumexp(S_VAL[nom][0]['identite'], 1), 1 - ACCEPTATION_CIBLE))
            L_seul = S_T[nom][0]['identite']
            RESULTATS[f'{ens} | {nom} seul, energie'], _ = mesurer(L_seul, torch.logsumexp(L_seul, 1), seuil, vrai)
    TABLEAU_TESTS = pd.DataFrame(RESULTATS).T
    print(TABLEAU_TESTS.to_string())
    for ens, t in TABLEAUX.items():
        print(f'\n{ens}, variante retenue, rejet {METHODE_REJET} :'); print(t)
    TABLEAU_TESTS.to_csv(f'{SORTIE}/tests.csv')
    json.dump(dict(methode_principale=f'{CHOIX.variante}, {METHODE_REJET}',
                   remarque='test_perou : premier test ; BananaLSD : indicatif (déjà consulté avec le modèle sans Pérou)',
                   resultats=RESULTATS), open(f'{SORTIE}/tests.json', 'w'), indent=2, ensure_ascii=False)
else:
    print('Tests non évalués (EVALUER_TESTS = False).')